## 1. Imports

In [ ]:
import pandas as pd

df_clean = pd.read_csv(
    r"C:\Users\carine\mars26_bds_immo\data\raw\df_clean.csv"
)

df_ML = df_clean.copy()

df_ML.info()

<class 'pandas.DataFrame'>
RangeIndex: 5818581 entries, 0 to 5818580
Data columns (total 19 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   annee                      int64  
 1   nature_mutation            str    
 2   valeur_fonciere            float64
 3   surface_reelle_bati        float64
 4   nombre_pieces_principales  float64
 5   surface_terrain            float64
 6   longitude                  float64
 7   latitude                   float64
 8   surface_log                float64
 9   type_local_code            int64  
 10  departement_code           int64  
 11  t                          int64  
 12  mois_sin                   float64
 13  mois_cos                   float64
 14  trimestre                  int64  
 15  is_vefa                    int64  
 16  surface_par_piece          float64
 17  anomalie_structure         int64  
 18  surface_terrain_log        float64
dtypes: float64(11), int64(7), str(1)
memory usage: 843.5 

In [ ]:
import numpy as np
df_ML["prix_log"] = np.log1p(df_ML["valeur_fonciere"])

In [ ]:
cols_to_drop = [
    "valeur_fonciere",     # fuite de données
    "t",                   # variable artificielle
    "type_local_code",     # encodage arbitraire
    "departement_code",    # encodage trompeur
    "anomalie_structure"   # variable trop rare
]

df_ML = df_ML.drop(columns=cols_to_drop, errors="ignore")

In [ ]:
df_ML.corr(numeric_only=True)["prix_log"].sort_values(ascending=False)

prix_log                     1.000000
surface_reelle_bati          0.402703
surface_log                  0.391852
nombre_pieces_principales    0.335508
surface_terrain_log          0.242724
surface_par_piece            0.141597
is_vefa                      0.059226
surface_terrain              0.057025
longitude                    0.013727
annee                        0.013550
trimestre                    0.006474
latitude                    -0.004782
mois_cos                    -0.011752
mois_sin                    -0.011777
Name: prix_log, dtype: float64

## 2. Encoding

In [ ]:
target = "prix_log"

X = df_ML.drop(columns=[target])
y = df_ML[target]

## 3. Split train/test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(4654864, 14) (1163717, 14)
(4654864,) (1163717,)


In [ ]:
X_train.isnull().sum().sum()

np.int64(3795904)

In [ ]:
X_train.isnull().sum().sort_values(ascending=False)

surface_terrain              1814252
surface_terrain_log          1814252
latitude                       83700
longitude                      83700
nature_mutation                    0
annee                              0
surface_reelle_bati                0
nombre_pieces_principales          0
mois_sin                           0
surface_log                        0
mois_cos                           0
trimestre                          0
is_vefa                            0
surface_par_piece                  0
dtype: int64

In [ ]:
df_ML[df_ML["surface_terrain"].isnull()].head()

,annee,nature_mutation,surface_reelle_bati,nombre_pieces_principales,surface_terrain,longitude,latitude,surface_log,mois_sin,mois_cos,trimestre,is_vefa,surface_par_piece,surface_terrain_log,prix_log
5,2020,Vente en l'état futur d'achèvement,61.0,3.0,NaN,4.838408,46.303181,4.127134,-0.5,-0.866025,3,1,20.333333,NaN,12.100551
6,2020,Vente en l'état futur d'achèvement,118.0,4.0,NaN,5.217651,46.208039,4.779123,-0.5,-0.866025,3,1,29.500000,NaN,12.793862
7,2020,Vente,62.0,3.0,NaN,5.219443,46.198796,4.143135,-0.5,-0.866025,3,0,20.666667,NaN,11.820418
10,2020,Vente en l'état futur d'achèvement,53.0,2.0,NaN,5.126002,46.337021,3.988984,-0.5,-0.866025,3,1,26.500000,NaN,11.728045
11,2020,Vente,111.0,2.0,NaN,5.224621,46.208263,4.718499,-0.5,-0.866025,3,0,55.500000,NaN,12.506181


In [ ]:
df_ML["has_terrain"] = df_ML["surface_terrain"].notna().astype(int)

In [ ]:
df_ML["surface_terrain"] = df_ML["surface_terrain"].fillna(0)
df_ML["surface_terrain_log"] = np.log1p(df_ML["surface_terrain"])

In [ ]:
df_ML["has_terrain"] = df_ML["surface_terrain"].notna().astype(int)

df_ML["surface_terrain"] = df_ML["surface_terrain"].fillna(0)

df_ML["surface_terrain_log"] = np.log1p(df_ML["surface_terrain"])

In [ ]:
df_ML["has_terrain"] = (df_ML["surface_terrain"].notna()).astype(int)

## Features SUR X_train ET X_test

In [ ]:
# Indicateur terrain
X_train["has_terrain"] = X_train["surface_terrain"].notna().astype(int)
X_test["has_terrain"] = X_test["surface_terrain"].notna().astype(int)

In [ ]:
#gestion des NaN terrain
X_train["surface_terrain"] = X_train["surface_terrain"].fillna(0)
X_test["surface_terrain"] = X_test["surface_terrain"].fillna(0)

In [ ]:
# transformation LOG
import numpy as np

X_train["surface_terrain_log"] = np.log1p(X_train["surface_terrain"])
X_test["surface_terrain_log"] = np.log1p(X_test["surface_terrain"])

In [ ]:
# fEATUres VEFA X TERRAIN
X_train["vefa_no_terrain"] = X_train["is_vefa"] * (X_train["has_terrain"] == 0).astype(int)
X_test["vefa_no_terrain"] = X_test["is_vefa"] * (X_test["has_terrain"] == 0).astype(int)

In [ ]:
X_train.isnull().sum().sum()
X_test.isnull().sum().sum()

np.int64(41274)

In [ ]:
X_train.isnull().sum().sort_values(ascending=False).head(10)

latitude                     83700
longitude                    83700
annee                            0
nature_mutation                  0
nombre_pieces_principales        0
surface_reelle_bati              0
surface_terrain                  0
surface_log                      0
mois_sin                         0
mois_cos                         0
dtype: int64

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
imputer = SimpleImputer(strategy="median")

In [ ]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

C:\Users\carine\AppData\Local\Temp\ipykernel_196\2590777664.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=["object"]).columns


In [ ]:
X_train = pd.get_dummies(X_train, columns=["nature_mutation"], drop_first=True)
X_test = pd.get_dummies(X_test, columns=["nature_mutation"], drop_first=True)

In [ ]:
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns
)

## 4. Models lightgbm

## 5. Evaluation

## 6. Relance modèle amélioré

## 7. Evaluation

## 8. Autres tunings